# Dataset 与 DataLoader

## 本节目标

当数据量变大时，不能总是一次性把全部样本送进模型。本节学习 `TensorDataset` 和 `DataLoader`，理解 batch、shuffle 和 mini-batch 训练。

完成本节后，你应该能够：

- 说清本节主题解决了什么问题。
- 读懂核心 PyTorch API 的输入输出。
- 运行最小代码示例并解释结果。
- 修改实验参数并观察变化。

## 背景问题

本节关注的问题是：当数据量变大时，不能总是一次性把全部样本送进模型。本节学习 `TensorDataset` 和 `DataLoader`，理解 batch、shuffle 和 mini-batch 训练。

学习时建议先运行代码，再回头看解释。对初学者来说，能观察 shape、loss、accuracy 或可视化结果，比只记概念更重要。

## 核心概念

- 输入和输出的 shape 必须明确。
- loss 用来描述模型当前预测和目标之间的差距。
- 优化器根据梯度更新模型参数。
- 实验观察需要结合曲线、数值和样本可视化。
- 调试时优先检查 shape、dtype、学习率和训练/评估模式。

这里的关键不是堆公式，而是把概念落实到可运行代码。

## 最小代码示例：构造 TensorDataset

这个代码块用于观察 `构造 TensorDataset`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
from torch.utils.data import TensorDataset, DataLoader

x = torch.randn(200, 3)
true_w = torch.tensor([[1.5], [-2.0], [0.7]])
y = x @ true_w + 0.1 * torch.randn(200, 1)

dataset = TensorDataset(x, y)
print("dataset size:", len(dataset))
print("first sample shapes:", dataset[0][0].shape, dataset[0][1].shape)

## 最小代码示例：使用 DataLoader

这个代码块用于观察 `使用 DataLoader`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
loader = DataLoader(dataset, batch_size=32, shuffle=True)

for batch_x, batch_y in loader:
    print("batch_x:", batch_x.shape)
    print("batch_y:", batch_y.shape)
    break

## 完整实验：mini-batch 训练

这个代码块用于观察 `mini-batch 训练`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
model = nn.Linear(3, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
losses = []

for epoch in range(50):
    total = 0.0
    for batch_x, batch_y in loader:
        pred = model(batch_x)
        loss = criterion(pred, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * batch_x.size(0)
    losses.append(total / len(dataset))

print("final loss:", losses[-1])

## 实验观察

DataLoader 让训练可以按 batch 进行。shuffle 通常用于训练集，有助于减少样本顺序对更新的影响。

从运行结果可以看到，代码输出不只是“能跑”，还可以帮助判断模型是否在按预期学习。建议把观察写进实验记录，而不是只保留最终准确率。

## 常见错误

- shape 不匹配：先打印输入、输出和标签 shape。
- dtype 不正确：分类标签通常是 `torch.long`，回归目标通常是 float。
- 忘记 `optimizer.zero_grad()`：梯度会累积。
- 学习率不合理：过大震荡，过小收敛慢。
- 评估阶段忘记 `model.eval()` 或 `torch.no_grad()`。

调试时可以先缩小数据集，确认模型能否在小样本上过拟合。

## 概念问答

**Q：为什么要从最小代码示例开始？**  
A：最小示例能隔离核心概念，降低同时排查多个问题的难度。

**Q：为什么每个实验都要看曲线或可视化？**  
A：单个最终数值不一定能解释训练过程，曲线和样本结果更容易暴露问题。

**Q：如果代码能运行但结果不好，先查什么？**  
A：优先检查数据、shape、label dtype、loss 使用方式和学习率。

## 延伸练习

- 把 batch_size 改成 8 和 128，对比 loss 曲线。
- 训练集使用 shuffle=False 观察差异。

这些练习不需要一次完成。建议每次只改一个变量，并记录改动前后的结果。

## 小结

本节把一个深度学习概念落到了可运行的 PyTorch 实验中。继续学习下一节前，建议确认自己能解释：输入是什么、模型做了什么、loss 如何计算、结果是否符合预期。